# ESG 문장 감성분석 실험

목적: ESG 관련 문장을 seed dictionary와 `v_1.ipynb`에서 가장 개선 폭이 컸던 `expanded_0_60` 사전으로 각각 추출한 뒤, 한국어 금융 감성분석 모델로 산출한 감성 feature가 KCGS ESG 등급과 어떤 관계를 보이는지 비교한다.

분석 방법: DART 사업보고서의 II, IV, VI 섹션을 기업-연도 단위로 묶고, seed dictionary와 `expanded_dictionary_theta_0_60.csv`를 각각 적용해 ESG 관련 문장을 추출한다. `snunlp/KR-FinBert-SC`로 각 ESG 문장의 긍정/부정/중립 점수를 계산한 뒤 기업-연도·사전별 긍정 ESG 문장 수, 부정 ESG 문장 수, 평균 감성점수를 만들고 Spearman 상관과 HC3 robust OLS로 등급과의 관계를 비교한다.

## 실행 결과 요약

### 데이터 및 문장 추출

| 항목 | 값 |
|---|---:|
| 분석 firm-year 수 | 378 |
| 분석 행 수(seed + expanded_0_60) | 756 |
| ESG 등급 결측 수 | 0 |
| 확장 사전 row 수 | 1,158 |
| seed 추출 ESG 문장 수 | 39,202 |
| expanded_0_60 추출 ESG 문장 수 | 40,231 |
| 전체 감성분석 문장 수 | 79,433 |

### 감성분석 결과

| sentiment_label | 문장 수 |
|---|---:|
| neutral | 69,724 |
| positive | 7,453 |
| negative | 2,256 |

### 사전별 평균 감성 feature

| dictionary | esg_sentence_count | positive_count | negative_count | neutral_count |
|---|---:|---:|---:|---:|
| expanded_0_60 | 106.431 | 9.997 | 3.032 | 93.402 |
| seed | 103.709 | 9.720 | 2.937 | 91.053 |

### Spearman 비교

| feature | expanded_0_60 rho | seed rho | expanded - seed |
|---|---:|---:|---:|
| esg_sentence_count | 0.661807 | 0.662474 | -0.000667 |
| esg_sentence_per_1000_words | -0.110656 | -0.117232 | 0.006576 |
| mean_esg_sentiment | 0.386327 | 0.395369 | -0.009042 |
| negative_esg_sentence_count | 0.288044 | 0.272602 | 0.015442 |
| negative_esg_share | -0.045754 | -0.052806 | 0.007052 |
| neutral_esg_sentence_count | 0.651207 | 0.651032 | 0.000175 |
| total_word_count | 0.653063 | 0.653063 | 0.000000 |

### OLS 결과

| dictionary | model | 주요 변수 | coef | p-value | R2 |
|---|---|---|---:|---:|---:|
| expanded_0_60 | M0_esg_sentence_count | esg_sentence_count | 0.852048 | 1.261896e-11 | 0.279697 |
| seed | M0_esg_sentence_count | esg_sentence_count | 0.851219 | 1.275368e-11 | 0.279153 |
| expanded_0_60 | M1_sentiment_counts | positive_esg_sentence_count | 0.778630 | 2.360244e-10 | 0.147380 |
| expanded_0_60 | M1_sentiment_counts | negative_esg_sentence_count | -0.238155 | 2.149366e-02 | 0.147380 |
| seed | M1_sentiment_counts | positive_esg_sentence_count | 0.803405 | 3.881610e-10 | 0.150126 |
| seed | M1_sentiment_counts | negative_esg_sentence_count | -0.267534 | 1.631929e-02 | 0.150126 |
| expanded_0_60 | M2_sentiment_counts_with_length | positive_esg_sentence_count | 0.298080 | 2.043591e-02 | 0.251189 |
| expanded_0_60 | M2_sentiment_counts_with_length | negative_esg_sentence_count | -0.236825 | 3.657207e-02 | 0.251189 |
| expanded_0_60 | M2_sentiment_counts_with_length | total_word_count | 0.706690 | 4.285191e-21 | 0.251189 |
| seed | M2_sentiment_counts_with_length | positive_esg_sentence_count | 0.318397 | 1.913862e-02 | 0.252555 |
| seed | M2_sentiment_counts_with_length | negative_esg_sentence_count | -0.253760 | 3.240051e-02 | 0.252555 |
| seed | M2_sentiment_counts_with_length | total_word_count | 0.700795 | 4.668376e-21 | 0.252555 |
| expanded_0_60 | M3_mean_sentiment_with_length | mean_esg_sentiment | 0.267474 | 3.394844e-04 | 0.311254 |
| expanded_0_60 | M3_mean_sentiment_with_length | esg_sentence_count | 0.522476 | 1.683157e-03 | 0.311254 |
| expanded_0_60 | M3_mean_sentiment_with_length | total_word_count | 0.255738 | 1.760963e-02 | 0.311254 |
| seed | M3_mean_sentiment_with_length | mean_esg_sentiment | 0.282547 | 1.365120e-04 | 0.313313 |
| seed | M3_mean_sentiment_with_length | esg_sentence_count | 0.513417 | 1.925315e-03 | 0.313313 |
| seed | M3_mean_sentiment_with_length | total_word_count | 0.256111 | 1.751579e-02 | 0.313313 |


In [1]:
from pathlib import Path
import re
import html
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)

FINAL_DIR = ROOT / "final"
DATA_DIR = ROOT / "data"
DART_DIR = DATA_DIR / "dart"

COMPANY_MASTER_PATH = first_existing([
    DATA_DIR / "company_master.csv",
    FINAL_DIR / "company_master.csv",
])
SEED_DICTIONARY_PATH = first_existing([
    DATA_DIR / "seed_dictionary.csv",
    FINAL_DIR / "seed_dictionary.csv",
])
FILING_INDEX_PATH = first_existing([
    DART_DIR / "filing_index.csv",
    FINAL_DIR / "filing_index.csv",
    DATA_DIR / "filing_index.csv",
])
RAW_XML_DIR = first_existing([
    DART_DIR / "raw_xml",
    FINAL_DIR / "raw_xml",
    ROOT / "raw_xml",
])
EXPANDED_DICTIONARY_PATH = first_existing([
    FINAL_DIR / "expanded_dictionaries" / "expanded_dictionary_theta_0_60.csv",
    DATA_DIR / "expanded_dictionaries" / "expanded_dictionary_theta_0_60.csv",
    ROOT / "expanded_dictionaries" / "expanded_dictionary_theta_0_60.csv",
], FINAL_DIR / "expanded_dictionaries" / "expanded_dictionary_theta_0_60.csv")

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}


def resolve_xml_path(xml_path_value):
    xml_path_text = str(xml_path_value).replace("\\", "/")
    candidates = [
        ROOT / xml_path_text,
        Path(xml_path_text),
        RAW_XML_DIR / Path(xml_path_text).name,
        FINAL_DIR / "raw_xml" / Path(xml_path_text).name,
        DART_DIR / "raw_xml" / Path(xml_path_text).name,
    ]
    return first_existing(candidates, candidates[0])


for path in [COMPANY_MASTER_PATH, SEED_DICTIONARY_PATH, FILING_INDEX_PATH, RAW_XML_DIR, EXPANDED_DICTIONARY_PATH]:
    print(path, "OK" if path.exists() else "MISSING")


Mounted at /content/drive
/content/drive/MyDrive/UD_26/data/company_master.csv OK
/content/drive/MyDrive/UD_26/final/seed_dictionary.csv OK
/content/drive/MyDrive/UD_26/final/filing_index.csv OK
/content/drive/MyDrive/UD_26/final/raw_xml OK
/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_60.csv OK


In [2]:
company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
filing_index = pd.read_csv(FILING_INDEX_PATH, dtype={"stock_code": "string", "rcept_no": "string"}, encoding="utf-8-sig")
expanded_df = pd.read_csv(EXPANDED_DICTIONARY_PATH, encoding="utf-8-sig") if EXPANDED_DICTIONARY_PATH.exists() else pd.DataFrame()

for df in [company_master, filing_index]:
    df["stock_code"] = df["stock_code"].str.zfill(6)

print("company_master:", company_master.shape)
print("seed_df:", seed_df.shape)
print("filing_index:", filing_index.shape)
print("expanded_df:", expanded_df.shape, "path:", EXPANDED_DICTIONARY_PATH)
display(company_master.head())
display(seed_df[["dimension", "seed_term", "pattern"]].head(10))


company_master: (381, 13)
seed_df: (30, 6)
filing_index: (381, 13)
expanded_df: (1158, 12) path: /content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_60.csv


,company_name,corp_code,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year
0,삼성전자,NaN,005930,전기전자,2022,11011,NaN,한국ESG기준원,A,A,A+,B+,2023
1,삼성전자,NaN,005930,전기전자,2023,11011,NaN,한국ESG기준원,B+,B+,A,B,2024
2,삼성전자,NaN,005930,전기전자,2024,11011,NaN,한국ESG기준원,A,B+,A+,B+,2025
3,BYC,NaN,001460,섬유/의류,2022,11011,NaN,한국ESG기준원,D,D,D,D,2023
4,BYC,NaN,001460,섬유/의류,2023,11011,NaN,한국ESG기준원,D,D,D,C,2024


,dimension,seed_term,pattern
0,E,탄소,탄소
1,E,온실가스,온실가스|GHG
2,E,탄소중립,탄소중립
3,E,넷제로,넷제로|net zero|net-zero
4,E,재생에너지,재생에너지|renewable energy
5,E,에너지,에너지
6,E,전력,전력|전력사용량
7,E,폐기물,폐기물
8,E,재활용,재활용|자원순환
9,E,폐수,폐수|수질|물관리


In [3]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def strip_tags_keep_space(xml_fragment):
    text = html.unescape(xml_fragment)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return unicodedata.normalize("NFKC", text)


title_re = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
main_title_re = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\.")
target_title_regex = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}


def extract_target_sections(xml_text):
    matches = list(title_re.finditer(xml_text))
    sections = []
    active = None

    for i, match in enumerate(matches):
        title = strip_tags_keep_space(match.group(1))
        next_start = matches[i + 1].start() if i + 1 < len(matches) else len(xml_text)
        body = xml_text[match.end():next_start]

        if main_title_re.search(title):
            active = None
            for section_name, pattern in target_title_regex.items():
                if re.search(pattern, title):
                    active = section_name
                    break
            if active is None:
                continue

        if active:
            section_text = strip_tags_keep_space(body)
            if section_text:
                sections.append({"section": active, "title": title, "text": section_text})

    return sections

rows = []

for _, row in filing_index.iterrows():
    xml_path = resolve_xml_path(row.get("xml_path", ""))
    if not xml_path.exists():
        continue
    xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
    for section in extract_target_sections(xml_text):
        rows.append({
            "stock_code": row["stock_code"],
            "company_name": row["company_name"],
            "fiscal_year": int(row["fiscal_year"]),
            "esg_year": int(row["esg_year"]),
            "rcept_no": row["rcept_no"],
            "section": section["section"],
            "section_text": normalize_text(section["text"]),
        })

section_df = pd.DataFrame(rows)
print("section rows:", section_df.shape)
display(section_df.groupby("section").size().rename("rows"))

section rows: (4155, 7)


,rows
section,
II. 사업의 내용,2643
IV. 이사의 경영진단 및 분석의견,378
VI. 이사회 등 회사의 기관에 관한 사항,1134


In [4]:
corpus_df = (
    section_df.groupby(["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no"], as_index=False)
    .agg(
        document=("section_text", " ".join),
        section_count=("section", "nunique"),
    )
)
corpus_df["document_norm"] = corpus_df["document"].map(normalize_text)
corpus_df["total_char_count"] = corpus_df["document_norm"].str.len()
corpus_df["total_word_count"] = corpus_df["document_norm"].str.split().str.len()

print("firm-year corpus rows:", len(corpus_df))
display(corpus_df[["stock_code", "company_name", "fiscal_year", "section_count", "total_word_count"]].head())

firm-year corpus rows: 378


,stock_code,company_name,fiscal_year,section_count,total_word_count
0,000020,동화약품,2022,3,7808
1,000020,동화약품,2023,3,8065
2,000020,동화약품,2024,3,8097
3,000040,KR모터스,2022,3,4201
4,000040,KR모터스,2023,3,4888


In [5]:
def pattern_from_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))
    terms = []
    seen = set()
    for value in values:
        value = normalize_text(value).strip()
        if not value or value.lower() == "nan" or value in seen:
            continue
        seen.add(value)
        terms.append(value)
    return terms


def build_seed_records(seed_df):
    records = []
    for _, row in seed_df.reset_index(drop=True).iterrows():
        dimension = str(row["dimension"]).strip()
        if dimension not in {"E", "S", "G"}:
            continue
        for term in pattern_from_terms(row):
            records.append({
                "dictionary_label": "seed",
                "dimension": dimension,
                "term": term,
                "regex": re.compile(re.escape(term), flags=re.I),
            })
    return records


def build_expanded_records(expanded_df, dictionary_label="expanded_0_60"):
    if expanded_df.empty:
        return []

    term_priority = [
        "term", "expanded_term", "candidate_term", "candidate", "word",
        "similar_word", "token", "pattern", "seed_term",
    ]
    term_cols = [col for col in term_priority if col in expanded_df.columns]
    if not term_cols:
        excluded = {"dimension", "source", "score", "similarity", "cosine", "rank", "threshold"}
        term_cols = [
            col for col in expanded_df.columns
            if col not in excluded and expanded_df[col].dtype == object
        ]

    records = []
    seen = set()
    for _, row in expanded_df.reset_index(drop=True).iterrows():
        dimension = str(row.get("dimension", "ESG")).strip()
        if dimension not in {"E", "S", "G"}:
            dimension = "ESG"
        for col in term_cols:
            value = row.get(col, "")
            if pd.isna(value):
                continue
            for term in str(value).split("|"):
                term = normalize_text(term).strip()
                if not term or term.lower() == "nan":
                    continue
                key = (dictionary_label, dimension, term)
                if key in seen:
                    continue
                seen.add(key)
                records.append({
                    "dictionary_label": dictionary_label,
                    "dimension": dimension,
                    "term": term,
                    "regex": re.compile(re.escape(term), flags=re.I),
                })
    return records


dictionary_records = build_seed_records(seed_df) + build_expanded_records(expanded_df)
if not dictionary_records:
    raise ValueError("사용 가능한 seed 또는 expanded dictionary record가 없습니다.")

dictionary_summary = (
    pd.DataFrame([{k: v for k, v in rec.items() if k != "regex"} for rec in dictionary_records])
    .groupby(["dictionary_label", "dimension"], as_index=False)
    .agg(terms=("term", "nunique"))
)
display(dictionary_summary)


def split_korean_sentences(text):
    text = normalize_text(text)
    rough = re.split(r"(?<=[\.\?\!])\s+|(?<=[다요음임함됨])\s+(?=[가-힣A-Z0-9])", text)
    return [normalize_text(sent) for sent in rough if len(normalize_text(sent)) >= 20]


def match_esg_sentence(sentence, dictionary_label):
    matched = []
    dimensions = set()
    for rec in dictionary_records:
        if rec["dictionary_label"] != dictionary_label:
            continue
        if rec["regex"].search(sentence):
            matched.append(rec["term"])
            dimensions.add(rec["dimension"])
    return sorted(dimensions), sorted(set(matched))


sentence_rows = []
dictionary_labels = sorted({rec["dictionary_label"] for rec in dictionary_records})

for _, row in corpus_df.iterrows():
    for sent in split_korean_sentences(row["document_norm"]):
        for dictionary_label in dictionary_labels:
            dimensions, matched_terms = match_esg_sentence(sent, dictionary_label)
            if not dimensions:
                continue
            sentence_rows.append({
                "dictionary_label": dictionary_label,
                "stock_code": row["stock_code"],
                "company_name": row["company_name"],
                "fiscal_year": row["fiscal_year"],
                "esg_year": row["esg_year"],
                "rcept_no": row["rcept_no"],
                "dimensions": ",".join(dimensions),
                "matched_terms": ", ".join(matched_terms[:12]),
                "sentence": sent,
            })

sentence_df = pd.DataFrame(sentence_rows)
print("ESG sentence rows:", sentence_df.shape)
display(sentence_df.groupby("dictionary_label")["sentence"].count().rename("sentences"))
display(sentence_df.head(10))


,dictionary_label,dimension,terms
0,expanded_0_60,E,948
1,expanded_0_60,G,164
2,expanded_0_60,S,218
3,seed,E,18
4,seed,G,17
5,seed,S,18


ESG sentence rows: (79433, 9)


,sentences
dictionary_label,
expanded_0_60,40231
seed,39202


,dictionary_label,stock_code,company_name,fiscal_year,esg_year,rcept_no,dimensions,matched_terms,sentence
0,expanded_0_60,000020,동화약품,2022,2022,20230315001100,S,교육,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다."
1,seed,000020,동화약품,2022,2022,20230315001100,S,교육,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다."
2,expanded_0_60,000020,동화약품,2022,2022,20230315001100,G,이사회,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.
3,seed,000020,동화약품,2022,2022,20230315001100,G,이사회,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.
4,expanded_0_60,000020,동화약품,2022,2022,20230315001100,G,"의결권, 주주","연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주..."
5,seed,000020,동화약품,2022,2022,20230315001100,G,"의결권, 주주","연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주..."
6,expanded_0_60,000020,동화약품,2022,2022,20230315001100,S,안전,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 ..."
7,seed,000020,동화약품,2022,2022,20230315001100,S,안전,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 ..."
8,expanded_0_60,000020,동화약품,2022,2022,20230315001100,G,윤리,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발..."
9,seed,000020,동화약품,2022,2022,20230315001100,G,윤리,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발..."


In [6]:
try:
    import torch
    from transformers import pipeline
except ImportError as exc:
    raise ImportError(
        "감성분석에는 transformers와 torch가 필요합니다. 설치 후 다시 실행하세요: uv pip install transformers torch"
    ) from exc

SENTIMENT_MODEL = "snunlp/KR-FinBert-SC"
DEVICE = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 64 if DEVICE == 0 else 32

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    truncation=True,
    max_length=256,
    device=DEVICE,
)

print("sentiment model:", SENTIMENT_MODEL)
print("device:", "cuda" if DEVICE == 0 else "cpu")
print("batch size:", BATCH_SIZE)
print("label map:", sentiment_pipe.model.config.id2label)


config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snunlp/KR-FinBert-SC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/406M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

sentiment model: snunlp/KR-FinBert-SC
device: cuda
batch size: 64
label map: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [7]:
def parse_sentiment_output(result):
    label_raw = str(result["label"])
    label = label_raw.upper().replace(" ", "_")
    score = float(result["score"])

    if label.startswith("LABEL_"):
        try:
            label_id = int(label.split("_", 1)[1])
            label = str(sentiment_pipe.model.config.id2label.get(label_id, label)).upper().replace(" ", "_")
        except ValueError:
            pass

    if any(token in label for token in ["NEG", "NEGATIVE", "부정", "하락", "악재"]):
        return "negative", -score
    if any(token in label for token in ["POS", "POSITIVE", "긍정", "상승", "호재"]):
        return "positive", score
    if any(token in label for token in ["NEU", "NEUTRAL", "중립"]):
        return "neutral", 0.0
    return label_raw.lower(), np.nan


sentences = sentence_df["sentence"].fillna("").tolist()
raw_results = sentiment_pipe(sentences, batch_size=BATCH_SIZE)

parsed = [parse_sentiment_output(result) for result in raw_results]
sentence_df["sentiment_label"] = [label for label, _ in parsed]
sentence_df["sentiment_score"] = [score for _, score in parsed]
sentence_df["positive_sentiment"] = (sentence_df["sentiment_label"] == "positive").astype(int)
sentence_df["negative_sentiment"] = (sentence_df["sentiment_label"] == "negative").astype(int)
sentence_df["neutral_sentiment"] = (sentence_df["sentiment_label"] == "neutral").astype(int)

display(sentence_df[["company_name", "fiscal_year", "matched_terms", "sentiment_label", "sentiment_score", "sentence"]].head(20))
display(sentence_df["sentiment_label"].value_counts(dropna=False).rename("sentences"))


,company_name,fiscal_year,matched_terms,sentiment_label,sentiment_score,sentence
0,동화약품,2022,교육,neutral,0.000000,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다."
1,동화약품,2022,교육,neutral,0.000000,"또한 브랜 드 인지도 및 대중매체 광고 활용, 전문적인 디테일 활동, 영업인력 전문교육을 통 한 영업력 강화 등을 통해 판매를 강화해 나가고 있습니다."
2,동화약품,2022,이사회,neutral,0.000000,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.
3,동화약품,2022,이사회,neutral,0.000000,연결기업의 재무부문은 이사회에서 승인된 위험관리 정책 및 절차에 따라 연결기업의 영업과 관련한 금융위험을 감시하고 관리하는 역할을 하고 있습니다.
4,동화약품,2022,"의결권, 주주",negative,-0.697641,"연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주..."
5,동화약품,2022,"의결권, 주주",negative,-0.697641,"연결실체의 종속회사의 전환상환우선주 발행에 따른 계약 상황은 아래와 같습니다.1) 메디쎄이 주식회사 발행 전환상환우선주 구 분 내 용 발행자 주식회사 메디쎄이 발행일자 2016-05-31 인수자 IBK기업은행, 아이비케이금융그룹 코넥스투자조합, 글로벌원밸류업전문사모투자신탁3호 주..."
6,동화약품,2022,안전,neutral,0.000000,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 ..."
7,동화약품,2022,안전,neutral,0.000000,"관계법령 또는 정부의 규제제약산업은 국민의 건강관리 및 질병의 예방, 치료, 처치 등을 위해 의약품을 개발·허가·제조 및 품질관리, 유통·판매하는 산업으로서 타 산업에 비해 많은 규제(안전성·유효성·안정성 확보, 약사법, 약가규제, 지적재산권 등)와 제약이 있습니다.최근 정부는 ..."
8,동화약품,2022,윤리,positive,0.978807,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발..."
9,동화약품,2022,윤리,positive,0.978807,"또한 정부의 약가 적정화 정책 및 한미 FTA체결, 약제비 적정화 방안 시행, 기등재의약품 목록 정비 사업, GMP기준 선진화 추진, 비윤리적 영업관행 금지 등급변하는 경쟁 환경속 에 각 제약사별 실적차별화가 예상되며 제품력, 영업력 및 브랜드인지도 등 경쟁요인과 더불어 신약개발..."


,sentences
sentiment_label,
neutral,69724
positive,7453
negative,2256


In [8]:
if sentence_df.empty:
    raise ValueError("ESG 문장이 추출되지 않았습니다. seed/expanded dictionary와 XML 섹션 추출을 확인하세요.")

sentiment_agg = (
    sentence_df.groupby(["dictionary_label", "stock_code", "fiscal_year"], as_index=False)
    .agg(
        esg_sentence_count=("sentence", "count"),
        positive_esg_sentence_count=("positive_sentiment", "sum"),
        negative_esg_sentence_count=("negative_sentiment", "sum"),
        neutral_esg_sentence_count=("neutral_sentiment", "sum"),
        mean_esg_sentiment=("sentiment_score", "mean"),
    )
)

base_panel = corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "total_word_count", "total_char_count", "section_count"]].copy()
dictionary_panel = pd.DataFrame({"dictionary_label": sorted(sentence_df["dictionary_label"].unique())})
feature_df = dictionary_panel.merge(base_panel, how="cross").merge(
    sentiment_agg,
    on=["dictionary_label", "stock_code", "fiscal_year"],
    how="left",
)

count_cols = [
    "esg_sentence_count",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "neutral_esg_sentence_count",
]
feature_df[count_cols] = feature_df[count_cols].fillna(0)
feature_df["mean_esg_sentiment"] = feature_df["mean_esg_sentiment"].fillna(0)
feature_df["positive_esg_share"] = feature_df["positive_esg_sentence_count"] / feature_df["esg_sentence_count"].replace(0, np.nan)
feature_df["negative_esg_share"] = feature_df["negative_esg_sentence_count"] / feature_df["esg_sentence_count"].replace(0, np.nan)
feature_df["neutral_esg_share"] = feature_df["neutral_esg_sentence_count"] / feature_df["esg_sentence_count"].replace(0, np.nan)
feature_df["esg_sentence_per_1000_words"] = 1000 * feature_df["esg_sentence_count"] / feature_df["total_word_count"].replace(0, np.nan)

display(feature_df.head())
display(
    feature_df.groupby("dictionary_label")[
        [
            "esg_sentence_count",
            "positive_esg_sentence_count",
            "negative_esg_sentence_count",
            "neutral_esg_sentence_count",
            "mean_esg_sentiment",
            "positive_esg_share",
            "negative_esg_share",
        ]
    ].mean()
)


,dictionary_label,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,esg_sentence_count,positive_esg_sentence_count,negative_esg_sentence_count,neutral_esg_sentence_count,mean_esg_sentiment,positive_esg_share,negative_esg_share,neutral_esg_share,esg_sentence_per_1000_words
0,expanded_0_60,000020,동화약품,2022,2022,20230315001100,7808,37953,3,46,2,2,42,0.013937,0.043478,0.043478,0.913043,5.891393
1,expanded_0_60,000020,동화약품,2023,2023,20240319000652,8065,38467,3,46,2,1,43,0.027842,0.043478,0.021739,0.934783,5.703658
2,expanded_0_60,000020,동화약품,2024,2024,20250318000739,8097,39259,3,50,2,0,48,0.039566,0.040000,0.000000,0.960000,6.175127
3,expanded_0_60,000040,KR모터스,2022,2022,20230322001182,4201,20136,3,17,1,0,16,0.058810,0.058824,0.000000,0.941176,4.046656
4,expanded_0_60,000040,KR모터스,2023,2023,20240321002062,4888,22402,3,14,1,1,12,0.033740,0.071429,0.071429,0.857143,2.864157


,esg_sentence_count,positive_esg_sentence_count,negative_esg_sentence_count,neutral_esg_sentence_count,mean_esg_sentiment,positive_esg_share,negative_esg_share
dictionary_label,,,,,,,
expanded_0_60,106.431217,9.997354,3.031746,93.402116,0.045009,0.073694,0.026143
seed,103.708995,9.719577,2.936508,91.052910,0.044901,0.073563,0.026070


In [9]:
grade_map = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}

grade_cols = ["stock_code", "fiscal_year", "industry", "esg_grade", "e_grade", "s_grade", "g_grade", "esg_year"]
grade_df = company_master[grade_cols].copy()
for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    grade_df[f"{col}_num"] = grade_df[col].map(grade_map)

feature_merge = feature_df.copy()
grade_merge = grade_df.copy()
for df in [feature_merge, grade_merge]:
    df["stock_code"] = df["stock_code"].astype("string").str.zfill(6)
    for col in ["fiscal_year", "esg_year"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
feature_merge["esg_year"] = feature_merge["fiscal_year"] + 1

analysis_df = feature_merge.merge(
    grade_merge,
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

print("analysis rows:", len(analysis_df))
print("missing esg grade:", analysis_df["esg_grade_num"].isna().sum())
display(analysis_df[["company_name", "stock_code", "fiscal_year", "esg_grade", "esg_grade_num", "positive_esg_sentence_count", "negative_esg_sentence_count", "mean_esg_sentiment"]].head())


analysis rows: 756
missing esg grade: 0


,company_name,stock_code,fiscal_year,esg_grade,esg_grade_num,positive_esg_sentence_count,negative_esg_sentence_count,mean_esg_sentiment
0,동화약품,000020,2022,C,1,2,2,0.013937
1,동화약품,000020,2023,C,1,2,1,0.027842
2,동화약품,000020,2024,C,1,2,0,0.039566
3,KR모터스,000040,2022,D,0,1,0,0.058810
4,KR모터스,000040,2023,D,0,1,1,0.033740


In [10]:
from scipy.stats import spearmanr

features_to_check = [
    "esg_sentence_count",
    "esg_sentence_per_1000_words",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "neutral_esg_sentence_count",
    "positive_esg_share",
    "negative_esg_share",
    "neutral_esg_share",
    "mean_esg_sentiment",
    "total_word_count",
]


def spearman_table(data, y_col, x_cols):
    rows = []
    for dictionary_label, group in data.groupby("dictionary_label"):
        for x_col in x_cols:
            tmp = group[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 3 or tmp[x_col].nunique() < 2:
                rows.append({
                    "dictionary_label": dictionary_label,
                    "feature": x_col,
                    "n": len(tmp),
                    "spearman_rho": np.nan,
                    "p_value": np.nan,
                })
                continue
            rho, p = spearmanr(tmp[x_col], tmp[y_col])
            rows.append({
                "dictionary_label": dictionary_label,
                "feature": x_col,
                "n": len(tmp),
                "spearman_rho": rho,
                "p_value": p,
            })
    return pd.DataFrame(rows).sort_values(["feature", "spearman_rho"], ascending=[True, False])


sentiment_spearman_df = spearman_table(analysis_df, "esg_grade_num", features_to_check)
display(sentiment_spearman_df)

if {"seed", "expanded_0_60"}.issubset(set(sentiment_spearman_df["dictionary_label"])):
    comparison_df = (
        sentiment_spearman_df
        .pivot(index="feature", columns="dictionary_label", values="spearman_rho")
        .reset_index()
    )
    comparison_df["rho_expanded_minus_seed"] = comparison_df["expanded_0_60"] - comparison_df["seed"]
    display(comparison_df.sort_values("rho_expanded_minus_seed", ascending=False))


,dictionary_label,feature,n,spearman_rho,p_value
10,seed,esg_sentence_count,378,0.662474,4.121648e-49
0,expanded_0_60,esg_sentence_count,378,0.661807,5.544702e-49
1,expanded_0_60,esg_sentence_per_1000_words,378,-0.110656,3.148576e-02
11,seed,esg_sentence_per_1000_words,378,-0.117232,2.263351e-02
18,seed,mean_esg_sentiment,378,0.395369,1.357668e-15
8,expanded_0_60,mean_esg_sentiment,378,0.386327,6.663356e-15
3,expanded_0_60,negative_esg_sentence_count,378,0.288044,1.177335e-08
13,seed,negative_esg_sentence_count,378,0.272602,7.254748e-08
6,expanded_0_60,negative_esg_share,378,-0.045754,3.750370e-01
16,seed,negative_esg_share,378,-0.052806,3.058425e-01


dictionary_label,feature,expanded_0_60,seed,rho_expanded_minus_seed
3,negative_esg_sentence_count,0.288044,0.272602,0.015442
4,negative_esg_share,-0.045754,-0.052806,0.007052
1,esg_sentence_per_1000_words,-0.110656,-0.117232,0.006576
5,neutral_esg_sentence_count,0.651207,0.651032,0.000174
9,total_word_count,0.653063,0.653063,0.000000
0,esg_sentence_count,0.661807,0.662474,-0.000667
6,neutral_esg_share,-0.300156,-0.298307,-0.001849
7,positive_esg_sentence_count,0.571918,0.574028,-0.002110
8,positive_esg_share,0.381632,0.386395,-0.004763
2,mean_esg_sentiment,0.386327,0.395369,-0.009042


In [11]:
import statsmodels.api as sm


def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def robust_ols(data, y_col, x_cols, min_n=3):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return pd.DataFrame({
            "coef": [np.nan],
            "std_err": [np.nan],
            "p_value": [np.nan],
            "r2": [np.nan],
            "n": [0],
            "note": [f"SKIPPED: missing columns: {missing_cols}"],
        }, index=["SKIPPED"])

    reg_df = data[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < min_n:
        return pd.DataFrame({
            "coef": [np.nan],
            "std_err": [np.nan],
            "p_value": [np.nan],
            "r2": [np.nan],
            "n": [int(len(reg_df))],
            "note": [f"SKIPPED: too few complete rows after dropna: {len(reg_df)}"],
        }, index=["SKIPPED"])
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "p_value": model.pvalues,
        "r2": model.rsquared,
        "n": int(model.nobs),
    })


model_specs = {
    "M0_esg_sentence_count": ["esg_sentence_count"],
    "M1_sentiment_counts": ["positive_esg_sentence_count", "negative_esg_sentence_count"],
    "M2_sentiment_counts_with_length": ["positive_esg_sentence_count", "negative_esg_sentence_count", "total_word_count"],
    "M3_mean_sentiment_with_length": ["mean_esg_sentiment", "esg_sentence_count", "total_word_count"],
}

for dictionary_label, group in analysis_df.groupby("dictionary_label"):
    print("\n" + "=" * 80)
    print("dictionary_label:", dictionary_label)
    for name, x_cols in model_specs.items():
        print("\n" + name)
        display(robust_ols(group, "esg_grade_num", x_cols))



dictionary_label: expanded_0_60

M0_esg_sentence_count


,coef,std_err,p_value,r2,n
const,2.619048,0.071083,3.493608e-297,0.279697,378
esg_sentence_count,0.852048,0.125802,1.261896e-11,0.279697,378



M1_sentiment_counts


,coef,std_err,p_value,r2,n
const,2.619048,0.077146,1.248770e-252,0.14738,378
positive_esg_sentence_count,0.778630,0.122893,2.360244e-10,0.14738,378
negative_esg_sentence_count,-0.238155,0.103582,2.149366e-02,0.14738,378



M2_sentiment_counts_with_length


...

,coef,std_err,p_value,r2,n
const,2.619048,0.077052,3.071535e-253,0.150126,378
positive_esg_sentence_count,0.803405,0.128366,3.881610e-10,0.150126,378
negative_esg_sentence_count,-0.267534,0.111394,1.631929e-02,0.150126,378



M2_sentiment_counts_with_length


,coef,std_err,p_value,r2,n
const,2.619048,0.072457,4.289394e-286,0.252555,378
positive_esg_sentence_count,0.318397,0.135903,1.913862e-02,0.252555,378
negative_esg_sentence_count,-0.253760,0.118611,3.240051e-02,0.252555,378
total_word_count,0.700795,0.074423,4.668376e-21,0.252555,378



M3_mean_sentiment_with_length


,coef,std_err,p_value,r2,n
const,2.619048,0.069611,8.665572e-310,0.313313,378
mean_esg_sentiment,0.282547,0.074074,1.365120e-04,0.313313,378
esg_sentence_count,0.513417,0.165537,1.925315e-03,0.313313,378
total_word_count,0.256111,0.107804,1.751579e-02,0.313313,378


In [12]:
OUTPUT_PATH = FINAL_DIR / "v_2_sentiment_analysis_df.csv"
analysis_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("saved:", OUTPUT_PATH)
print("shape:", analysis_df.shape)

saved: /content/drive/MyDrive/UD_26/final/v_2_sentiment_analysis_df.csv
shape: (756, 27)
